# ICT-Greffe2 — Le quadruplet $(A, F, r, \pi)$ : rendre « élargir son espace » testable

**Navigation** : [Index](README.md) | [<< ICT-30 Inhibited Invention](ICT-30-InhibitedInvention-Python.ipynb) | [Version Python](ICT-Greffe2-EspaceAtteignable-Python.ipynb) | [Version C#](ICT-Greffe2-EspaceAtteignable-CSharp.ipynb)

**Greffe 2 de la strate 7** (#13568). La strate 7 dit que les agents « élargissent leur espace ». La formule est nébuleuse — ce notebook la rend **mesurable** en nommant quatre objets, et en montrant que chacun des quatre peut bouger **indépendamment** des trois autres. Le banc consomme deux instruments déjà livrés par la série [Planners](../../SymbolicAI/Planners/README.md) :

- **[Planners-5c](../../SymbolicAI/Planners/02-Classical/Planners-05c-Differentiel-Atteignabilite-Python.ipynb)** — le *différentiel d'atteignabilité* : ce que l'ajout d'une primitive rend possible (bras **élargissement**) ;
- **[Planners-10b](../../SymbolicAI/Planners/04-NeuroSymbolic/Planners-10b-LLM-Space-Reducer-Python.ipynb)** — le LLM comme *réducteur d'espace* (bras **restriction**, moteur [aicpp](https://github.com/Julien-Livet/aicpp)).

Aucun des deux ne nomme le quadruplet complet ; aucun ne mesure le **noyau** (ce que l'ajout rend *inatteignable*). C'est ce que cette greffe ajoute — sur le **même domaine** que Planners-5c, en consommant son socle sans le re-dériver.

> **Statut épistémique** — **Sans verdict à ce jour** : aucune ligne de la [matrice de dissociations](../../../docs/ict/dissociations-matrix.md) ne concerne ce notebook ; son statut épistémique sera porté par la matrice le cas échéant.

## Le quadruplet, nommé

| Objet | Définition opérationnelle | Se mesure par |
|---|---|---|
| $A$ — espace ambiant | l'ensemble des états atteignables compte tenu du **vocabulaire** de primitives $V$ | BFS exhaustif : $A(V) = \{s : \exists \pi,\ run\ \pi\ s_0 \to s\}$ |
| $F$ — espace admissible | la part de $A$ que le **solveur** explore effectivement sous son budget $B$ (expansions) | BFS tronqué à $B$ expansions |
| $r$ — liberté résiduelle | le rapport $\lvert F_B\rvert / \lvert A\rvert$ : combien de l'ambiant reste jouable à budget donné | ratio direct |
| $\pi$ — politique | l'**ordre d'expansion** des successeurs (ici : déterministe, paramétrable) | coût en expansions de la première découverte d'un but |

Pourquoi **quatre** objets : dire « l'agent a progressé » sans dire *lequel* des quatre a bougé ne dit rien. Élargir $A$ sans vérifier $F$ peut **contracter** $r$ (l'espace croît plus vite que le budget) ; changer $\pi$ améliore la navigation sans toucher ni $A$ ni $F$. Les trois cas de figure de la strate 7 (contraction certifiée / navigation améliorée / élargissement réel) deviennent trois **signatures différentes** sur le quadruplet — et la greffe 1 de #13565 (Beck-Fiala) fournira le témoin de contraction *certifiée* quand il sera livré.

## Le socle STRIPS — copie de Planners-5c, garanties du lake

Le moteur est celui de Planners-5c (copie fidèle, non modifiée). Il consomme le lake `planning_lean` **sans le re-dériver** : les trois résultats certifiés sans `sorry` qui fondent la lecture sont cités en garantie dans la section suivante.

In [2]:
use std::collections::{BTreeSet, HashMap, VecDeque};

// Operateur STRIPS : preconditions, ajouts, suppressions (frozenset -> BTreeSet)
#[derive(Clone)]
struct Op {
    nom: String,
    precond: BTreeSet<String>,
    ajouts: BTreeSet<String>,
    dels: BTreeSet<String>,
}

impl Op {
    // Applicable si toutes les preconditions sont dans l'etat
    fn applicable(&self, etat: &BTreeSet<String>) -> bool {
        self.precond.is_subset(etat)
    }
}

// Successeurs : tri par la cle de politique, puis enfant = (etat ∪ ajouts) − dels
fn successeurs(etat: &BTreeSet<String>, ops: &[Op], cle: &impl Fn(&Op) -> (u32, String)) -> Vec<(Op, BTreeSet<String>)> {
    let mut tried: Vec<&Op> = ops.iter().collect();
    tried.sort_by_key(|o| cle(o));
    let mut out = Vec::new();
    for op in tried {
        if op.applicable(etat) {
            let mut enfant = etat.clone();
            enfant.extend(op.ajouts.iter().cloned());
            for d in &op.dels {
                enfant.remove(d);
            }
            out.push((op.clone(), enfant));
        }
    }
    out
}

// BFS budgete : budget teste AVANT le pop ; dist = HashMap (les etats ne sont jamais iteres)
fn bfs_budget(initial: &BTreeSet<String>, ops: &[Op], budget: Option<usize>, cle: &impl Fn(&Op) -> (u32, String)) -> (HashMap<BTreeSet<String>, usize>, usize) {
    let mut dist: HashMap<BTreeSet<String>, usize> = HashMap::new();
    dist.insert(initial.clone(), 0);
    let mut file: VecDeque<BTreeSet<String>> = VecDeque::new();
    file.push_back(initial.clone());
    let mut expansions = 0;
    while !file.is_empty() && budget.map_or(true, |b| expansions < b) {
        let s = file.pop_front().unwrap();
        expansions += 1;
        for (_op, t) in successeurs(&s, ops, cle) {
            if !dist.contains_key(&t) {
                dist.insert(t.clone(), dist[&s] + 1);
                file.push_back(t);
            }
        }
    }
    (dist, expansions)
}

println!("Moteur STRIPS charge (copie Planners-5c) : step + BFS budgete + politique parametrable.");

Moteur STRIPS charge (copie Planners-5c) : step + BFS budgete + politique parametrable.


### Ce que le lake garantit — et pourquoi il n'est pas re-dérivé ici

Le lake `planning_lean` (consommé par Planners-5b/5c) prouve, sans `sorry` :

| Résultat certifié | Lieu | Ce qu'il garantit ici |
|---|---|---|
| $run\ \pi\ s \subseteq runR\ \pi\ s$ (toute trajectoire réelle est une trajectoire relâchée) | `Planning/Relaxation.lean:56` | la relaxation ne peut pas « sauver » un but : si $g$ est inatteignable en relâché, il l'est réellement |
| $step \subseteq stepR$ | `Planning/Strips.lean:58` | le lemme central de la monotonie relâchée |
| admissibilité $h^+ \le h^*$ | `Planning/Admissibility.lean:41` | les minorants heuristiques ne mentent jamais sur la distance |

L'interdit de #13568 est explicite : **réimplémenter un espace d'états, une relaxation ou une heuristique dans `ict/` alors que `planning_lean` les prouve**. Ce notebook n'en réimplémente aucune : le BFS est l'*instrument de mesure* de $A$ et $F$ (c'est le rôle qu'il a déjà dans Planners-5c), les garanties sont citées, pas re-prouvées.

## Le domaine : deux rives, un gouffre, un radeau qui n'existe pas encore

Domaine de Planners-5c, copie fidèle : deux rives (colonnes 0-2 et 4-5), un gouffre (colonne 3, infranchissable), une planchette, une clé, un trésor. Trois buts : $g_1$ (balise sur la rive droite), $g_2$ (détenir le trésor), $g_3$ (détenir la clé). Dans le vocabulaire de base $A_t$, la rive droite est **inatteignable** — donc $g_1$ et $g_2$ aussi.

In [3]:
use std::collections::BTreeSet;

#[derive(Clone)]
struct Op {
    nom: String,
    precond: BTreeSet<String>,
    ajouts: BTreeSet<String>,
    dels: BTreeSet<String>,
}

impl Op {
    fn applicable(&self, etat: &BTreeSet<String>) -> bool {
        self.precond.is_subset(etat)
    }
}

// Les 15 cases en ordre c-majeur r-minor — colonne 3 = gouffre
let cases: Vec<(i64, i64)> = {
    let cols_gauche = [0i64, 1, 2];
    let cols_droite = [4i64, 5];
    let rangs = [0i64, 1, 2];
    let mut s: BTreeSet<(i64, i64)> = BTreeSet::new();
    for &c in cols_gauche.iter().chain(cols_droite.iter()) {
        for &r in &rangs {
            s.insert((c, r));
        }
    }
    s.into_iter().collect()
};

fn prop(c: i64, r: i64) -> String {
    format!("at-x{}-y{}", c, r)
}

let initial: BTreeSet<String> = {
    let mut s = BTreeSet::new();
    s.insert(prop(0, 0));
    s.insert("plank-at-x1-y1".to_string());
    s.insert("key-at-x2-y0".to_string());
    s.insert("treasure-at-x4-y1".to_string());
    s
};

fn ops_de_base(cases: &[(i64, i64)]) -> Vec<Op> {
    let mut ops = Vec::new();
    for &(c, r) in cases {
        for (dc, dr, nom_dir) in [(1i64, 0i64, "est"), (-1, 0, "ouest"), (0, 1, "nord"), (0, -1, "sud")] {
            let cible = (c + dc, r + dr);
            if cases.contains(&cible) {
                ops.push(Op {
                    nom: format!("move-{}-x{}-y{}", nom_dir, c, r),
                    precond: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s },
                    ajouts: { let mut s = BTreeSet::new(); s.insert(prop(cible.0, cible.1)); s },
                    dels: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s },
                });
            }
        }
    }
    for (objet, cell) in [("plank", (1i64, 1i64)), ("key", (2i64, 0i64)), ("treasure", (4i64, 1i64))] {
        let (c, r) = cell;
        ops.push(Op {
            nom: format!("pickup-{}", objet),
            precond: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s.insert(format!("{}-at-x{}-y{}", objet, c, r)); s },
            ajouts: { let mut s = BTreeSet::new(); s.insert(format!("holding-{}", objet)); s },
            dels: { let mut s = BTreeSet::new(); s.insert(format!("{}-at-x{}-y{}", objet, c, r)); s },
        });
    }
    ops
}

fn py_list(items: &[&str]) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}

let buts: Vec<(&str, BTreeSet<String>)> = vec![
    ("g1: balise (5,2)", { let mut s = BTreeSet::new(); s.insert(prop(5, 2)); s }),
    ("g2: detenir le tresor", { let mut s = BTreeSet::new(); s.insert("holding-treasure".to_string()); s }),
    ("g3: detenir la cle", { let mut s = BTreeSet::new(); s.insert("holding-key".to_string()); s }),
];

println!("Cases existantes : {} (colonne 3 = gouffre)", cases.len());
let noms: Vec<&str> = buts.iter().map(|(n, _)| *n).collect();
println!("Buts : {}", py_list(&noms));

Cases existantes : 15 (colonne 3 = gouffre)


Buts : ['g1: balise (5,2)', 'g2: detenir le tresor', 'g3: detenir la cle']


In [4]:
use std::collections::BTreeSet;

#[derive(Clone)]
struct Op {
    nom: String,
    precond: BTreeSet<String>,
    ajouts: BTreeSet<String>,
    dels: BTreeSet<String>,
}

impl Op {
    fn applicable(&self, etat: &BTreeSet<String>) -> bool {
        self.precond.is_subset(etat)
    }
}

// Les 15 cases en ordre c-majeur r-minor — colonne 3 = gouffre
let cases: Vec<(i64, i64)> = {
    let cols_gauche = [0i64, 1, 2];
    let cols_droite = [4i64, 5];
    let rangs = [0i64, 1, 2];
    let mut s: BTreeSet<(i64, i64)> = BTreeSet::new();
    for &c in cols_gauche.iter().chain(cols_droite.iter()) {
        for &r in &rangs {
            s.insert((c, r));
        }
    }
    s.into_iter().collect()
};

fn prop(c: i64, r: i64) -> String {
    format!("at-x{}-y{}", c, r)
}

fn ops_de_base(cases: &[(i64, i64)]) -> Vec<Op> {
    let mut ops = Vec::new();
    for &(c, r) in cases {
        for (dc, dr, nom_dir) in [(1i64, 0i64, "est"), (-1, 0, "ouest"), (0, 1, "nord"), (0, -1, "sud")] {
            let cible = (c + dc, r + dr);
            if cases.contains(&cible) {
                ops.push(Op {
                    nom: format!("move-{}-x{}-y{}", nom_dir, c, r),
                    precond: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s },
                    ajouts: { let mut s = BTreeSet::new(); s.insert(prop(cible.0, cible.1)); s },
                    dels: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s },
                });
            }
        }
    }
    for (objet, cell) in [("plank", (1i64, 1i64)), ("key", (2i64, 0i64)), ("treasure", (4i64, 1i64))] {
        let (c, r) = cell;
        ops.push(Op {
            nom: format!("pickup-{}", objet),
            precond: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s.insert(format!("{}-at-x{}-y{}", objet, c, r)); s },
            ajouts: { let mut s = BTreeSet::new(); s.insert(format!("holding-{}", objet)); s },
            dels: { let mut s = BTreeSet::new(); s.insert(format!("{}-at-x{}-y{}", objet, c, r)); s },
        });
    }
    ops
}

fn ops_avec_radeau(cases: &[(i64, i64)]) -> Vec<Op> {
    let mut ops = ops_de_base(cases);
    for r in [0i64, 1, 2] {
        ops.push(Op {
            nom: format!("raft-across-y{}", r),
            precond: { let mut s = BTreeSet::new(); s.insert(prop(2, r)); s.insert("holding-plank".to_string()); s },
            ajouts: { let mut s = BTreeSet::new(); s.insert(prop(4, r)); s },
            dels: { let mut s = BTreeSet::new(); s.insert(prop(2, r)); s.insert("holding-plank".to_string()); s },
        });
    }
    ops
}

fn ops_avec_jeter(base: Vec<Op>) -> Vec<Op> {
    let mut ops = base;
    ops.push(Op {
        nom: "discard-plank".to_string(),
        precond: { let mut s = BTreeSet::new(); s.insert("holding-plank".to_string()); s },
        ajouts: BTreeSet::new(),
        dels: { let mut s = BTreeSet::new(); s.insert("holding-plank".to_string()); s },
    });
    ops
}

let a_t = ops_de_base(&cases);           // vocabulaire de base
let ster = ops_avec_jeter(a_t.clone());  // controle negatif : ajout sterile
let a_t1 = ops_avec_radeau(&cases);     // elargissement reel
let t1s = ops_avec_jeter(a_t1.clone()); // elargissement + pieges

let vocabs: Vec<(&str, Vec<Op>)> = vec![
    ("A_t", a_t),
    ("A_t+discard", ster),
    ("A_t1 (+raft)", a_t1),
    ("A_t1+discard", t1s),
];

for (nom, v) in &vocabs {
    println!("{:<16} |V| = {} operateurs", nom, v.len());
};

A_t              |V| = 41 operateurs


A_t+discard      |V| = 42 operateurs


A_t1 (+raft)     |V| = 44 operateurs


A_t1+discard     |V| = 45 operateurs


## $A$ — le différentiel d'atteignabilité **système**

Première mesure : l'espace ambiant $A(V)$ en entier (BFS exhaustif), et les buts atteignables dans chacun.

In [5]:
use std::collections::{BTreeSet, HashMap, VecDeque};

#[derive(Clone)]
struct Op {
    nom: String,
    precond: BTreeSet<String>,
    ajouts: BTreeSet<String>,
    dels: BTreeSet<String>,
}

impl Op {
    fn applicable(&self, etat: &BTreeSet<String>) -> bool {
        self.precond.is_subset(etat)
    }
}

fn successeurs(etat: &BTreeSet<String>, ops: &[Op], cle: &impl Fn(&Op) -> (u32, String)) -> Vec<(Op, BTreeSet<String>)> {
    let mut tried: Vec<&Op> = ops.iter().collect();
    tried.sort_by_key(|o| cle(o));
    let mut out = Vec::new();
    for op in tried {
        if op.applicable(etat) {
            let mut enfant = etat.clone();
            enfant.extend(op.ajouts.iter().cloned());
            for d in &op.dels {
                enfant.remove(d);
            }
            out.push((op.clone(), enfant));
        }
    }
    out
}

fn bfs_budget(initial: &BTreeSet<String>, ops: &[Op], budget: Option<usize>, cle: &impl Fn(&Op) -> (u32, String)) -> (HashMap<BTreeSet<String>, usize>, usize) {
    let mut dist: HashMap<BTreeSet<String>, usize> = HashMap::new();
    dist.insert(initial.clone(), 0);
    let mut file: VecDeque<BTreeSet<String>> = VecDeque::new();
    file.push_back(initial.clone());
    let mut expansions = 0;
    while !file.is_empty() && budget.map_or(true, |b| expansions < b) {
        let s = file.pop_front().unwrap();
        expansions += 1;
        for (_op, t) in successeurs(&s, ops, cle) {
            if !dist.contains_key(&t) {
                dist.insert(t.clone(), dist[&s] + 1);
                file.push_back(t);
            }
        }
    }
    (dist, expansions)
}

// Domaine
let cases: Vec<(i64, i64)> = {
    let cols_gauche = [0i64, 1, 2];
    let cols_droite = [4i64, 5];
    let rangs = [0i64, 1, 2];
    let mut s: BTreeSet<(i64, i64)> = BTreeSet::new();
    for &c in cols_gauche.iter().chain(cols_droite.iter()) {
        for &r in &rangs {
            s.insert((c, r));
        }
    }
    s.into_iter().collect()
};

fn prop(c: i64, r: i64) -> String {
    format!("at-x{}-y{}", c, r)
}

let initial: BTreeSet<String> = {
    let mut s = BTreeSet::new();
    s.insert(prop(0, 0));
    s.insert("plank-at-x1-y1".to_string());
    s.insert("key-at-x2-y0".to_string());
    s.insert("treasure-at-x4-y1".to_string());
    s
};

fn ops_de_base(cases: &[(i64, i64)]) -> Vec<Op> {
    let mut ops = Vec::new();
    for &(c, r) in cases {
        for (dc, dr, nom_dir) in [(1i64, 0i64, "est"), (-1, 0, "ouest"), (0, 1, "nord"), (0, -1, "sud")] {
            let cible = (c + dc, r + dr);
            if cases.contains(&cible) {
                ops.push(Op {
                    nom: format!("move-{}-x{}-y{}", nom_dir, c, r),
                    precond: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s },
                    ajouts: { let mut s = BTreeSet::new(); s.insert(prop(cible.0, cible.1)); s },
                    dels: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s },
                });
            }
        }
    }
    for (objet, cell) in [("plank", (1i64, 1i64)), ("key", (2i64, 0i64)), ("treasure", (4i64, 1i64))] {
        let (c, r) = cell;
        ops.push(Op {
            nom: format!("pickup-{}", objet),
            precond: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s.insert(format!("{}-at-x{}-y{}", objet, c, r)); s },
            ajouts: { let mut s = BTreeSet::new(); s.insert(format!("holding-{}", objet)); s },
            dels: { let mut s = BTreeSet::new(); s.insert(format!("{}-at-x{}-y{}", objet, c, r)); s },
        });
    }
    ops
}

fn ops_avec_radeau(cases: &[(i64, i64)]) -> Vec<Op> {
    let mut ops = ops_de_base(cases);
    for r in [0i64, 1, 2] {
        ops.push(Op {
            nom: format!("raft-across-y{}", r),
            precond: { let mut s = BTreeSet::new(); s.insert(prop(2, r)); s.insert("holding-plank".to_string()); s },
            ajouts: { let mut s = BTreeSet::new(); s.insert(prop(4, r)); s },
            dels: { let mut s = BTreeSet::new(); s.insert(prop(2, r)); s.insert("holding-plank".to_string()); s },
        });
    }
    ops
}

fn ops_avec_jeter(base: Vec<Op>) -> Vec<Op> {
    let mut ops = base;
    ops.push(Op {
        nom: "discard-plank".to_string(),
        precond: { let mut s = BTreeSet::new(); s.insert("holding-plank".to_string()); s },
        ajouts: BTreeSet::new(),
        dels: { let mut s = BTreeSet::new(); s.insert("holding-plank".to_string()); s },
    });
    ops
}

let a_t = ops_de_base(&cases);
let ster = ops_avec_jeter(a_t.clone());
let a_t1 = ops_avec_radeau(&cases);
let t1s = ops_avec_jeter(a_t1.clone());

let vocabs: Vec<(&str, Vec<Op>)> = vec![
    ("A_t", a_t),
    ("A_t+discard", ster),
    ("A_t1 (+raft)", a_t1),
    ("A_t1+discard", t1s),
];

let buts: Vec<(&str, BTreeSet<String>)> = vec![
    ("g1: balise (5,2)", { let mut s = BTreeSet::new(); s.insert(prop(5, 2)); s }),
    ("g2: detenir le tresor", { let mut s = BTreeSet::new(); s.insert("holding-treasure".to_string()); s }),
    ("g3: detenir la cle", { let mut s = BTreeSet::new(); s.insert("holding-key".to_string()); s }),
];

fn py_list(items: &[&str]) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}

{
    let pi_lex = |o: &Op| (0u32, o.nom.clone());

    let mut atteint: HashMap<&str, HashMap<BTreeSet<String>, usize>> = HashMap::new();
    for (nom, v) in &vocabs {
        let (dist, _) = bfs_budget(&initial, v, None, &pi_lex);
        atteint.insert(nom, dist);
    }

    for (nom, _) in &vocabs {
        let dist = &atteint[nom];
        let buts_ok: Vec<&str> = buts.iter()
            .filter(|(b, cond)| dist.keys().any(|s| cond.is_subset(s)))
            .map(|(b, _)| *b)
            .collect();
        let mut prefixes: Vec<&str> = buts_ok.iter().map(|b| &b[..2]).collect();
        prefixes.sort();
        println!("{:<16} |A| = {:>4} etats   buts atteignables : {}/3 {}", nom, dist.len(), buts_ok.len(), py_list(&prefixes));
    }
};

A_t              |A| =   36 etats   buts atteignables : 1/3 ['g3']


A_t+discard      |A| =   54 etats   buts atteignables : 1/3 ['g3']


A_t1 (+raft)     |A| =   60 etats   buts atteignables : 3/3 ['g1', 'g2', 'g3']


A_t1+discard     |A| =   78 etats   buts atteignables : 3/3 ['g1', 'g2', 'g3']


### Lecture : le différentiel non nul (critère 2 de #13568)

`+raft` fait passer de **1/3** à **3/3** buts : $g_1$ (balise) et $g_2$ (trésor) sont **nouvellement atteignables**. C'est l'élargissement réel au niveau du **système** — exactement le différentiel de Planners-5c, désormais exprimé sur le quadruplet : $A$ croît (36 → 60 états) **et** les buts bougent.

## Contrôle négatif : la primitive stérile (critère 3)

Un différentiel *toujours positif* ne mesure pas, il décore. Le contrôle obligatoire : un ajout de primitive qui **n'élargit rien**. `discard-plank` (jeter la planchette) est le candidat — dans $A_t$ il n'y a rien à traverser, la planchette ne sert à rien, la détruire ne peut pas aider.

In [6]:
use std::collections::{BTreeSet, HashMap, VecDeque};

#[derive(Clone)]
struct Op {
    nom: String,
    precond: BTreeSet<String>,
    ajouts: BTreeSet<String>,
    dels: BTreeSet<String>,
}

impl Op {
    fn applicable(&self, etat: &BTreeSet<String>) -> bool {
        self.precond.is_subset(etat)
    }
}

fn successeurs(etat: &BTreeSet<String>, ops: &[Op], cle: &impl Fn(&Op) -> (u32, String)) -> Vec<(Op, BTreeSet<String>)> {
    let mut tried: Vec<&Op> = ops.iter().collect();
    tried.sort_by_key(|o| cle(o));
    let mut out = Vec::new();
    for op in tried {
        if op.applicable(etat) {
            let mut enfant = etat.clone();
            enfant.extend(op.ajouts.iter().cloned());
            for d in &op.dels {
                enfant.remove(d);
            }
            out.push((op.clone(), enfant));
        }
    }
    out
}

fn bfs_budget(initial: &BTreeSet<String>, ops: &[Op], budget: Option<usize>, cle: &impl Fn(&Op) -> (u32, String)) -> (HashMap<BTreeSet<String>, usize>, usize) {
    let mut dist: HashMap<BTreeSet<String>, usize> = HashMap::new();
    dist.insert(initial.clone(), 0);
    let mut file: VecDeque<BTreeSet<String>> = VecDeque::new();
    file.push_back(initial.clone());
    let mut expansions = 0;
    while !file.is_empty() && budget.map_or(true, |b| expansions < b) {
        let s = file.pop_front().unwrap();
        expansions += 1;
        for (_op, t) in successeurs(&s, ops, cle) {
            if !dist.contains_key(&t) {
                dist.insert(t.clone(), dist[&s] + 1);
                file.push_back(t);
            }
        }
    }
    (dist, expansions)
}

// Domaine
let cases: Vec<(i64, i64)> = {
    let cols_gauche = [0i64, 1, 2];
    let cols_droite = [4i64, 5];
    let rangs = [0i64, 1, 2];
    let mut s: BTreeSet<(i64, i64)> = BTreeSet::new();
    for &c in cols_gauche.iter().chain(cols_droite.iter()) {
        for &r in &rangs {
            s.insert((c, r));
        }
    }
    s.into_iter().collect()
};

fn prop(c: i64, r: i64) -> String {
    format!("at-x{}-y{}", c, r)
}

let initial: BTreeSet<String> = {
    let mut s = BTreeSet::new();
    s.insert(prop(0, 0));
    s.insert("plank-at-x1-y1".to_string());
    s.insert("key-at-x2-y0".to_string());
    s.insert("treasure-at-x4-y1".to_string());
    s
};

fn ops_de_base(cases: &[(i64, i64)]) -> Vec<Op> {
    let mut ops = Vec::new();
    for &(c, r) in cases {
        for (dc, dr, nom_dir) in [(1i64, 0i64, "est"), (-1, 0, "ouest"), (0, 1, "nord"), (0, -1, "sud")] {
            let cible = (c + dc, r + dr);
            if cases.contains(&cible) {
                ops.push(Op {
                    nom: format!("move-{}-x{}-y{}", nom_dir, c, r),
                    precond: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s },
                    ajouts: { let mut s = BTreeSet::new(); s.insert(prop(cible.0, cible.1)); s },
                    dels: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s },
                });
            }
        }
    }
    for (objet, cell) in [("plank", (1i64, 1i64)), ("key", (2i64, 0i64)), ("treasure", (4i64, 1i64))] {
        let (c, r) = cell;
        ops.push(Op {
            nom: format!("pickup-{}", objet),
            precond: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s.insert(format!("{}-at-x{}-y{}", objet, c, r)); s },
            ajouts: { let mut s = BTreeSet::new(); s.insert(format!("holding-{}", objet)); s },
            dels: { let mut s = BTreeSet::new(); s.insert(format!("{}-at-x{}-y{}", objet, c, r)); s },
        });
    }
    ops
}

fn ops_avec_radeau(cases: &[(i64, i64)]) -> Vec<Op> {
    let mut ops = ops_de_base(cases);
    for r in [0i64, 1, 2] {
        ops.push(Op {
            nom: format!("raft-across-y{}", r),
            precond: { let mut s = BTreeSet::new(); s.insert(prop(2, r)); s.insert("holding-plank".to_string()); s },
            ajouts: { let mut s = BTreeSet::new(); s.insert(prop(4, r)); s },
            dels: { let mut s = BTreeSet::new(); s.insert(prop(2, r)); s.insert("holding-plank".to_string()); s },
        });
    }
    ops
}

fn ops_avec_jeter(base: Vec<Op>) -> Vec<Op> {
    let mut ops = base;
    ops.push(Op {
        nom: "discard-plank".to_string(),
        precond: { let mut s = BTreeSet::new(); s.insert("holding-plank".to_string()); s },
        ajouts: BTreeSet::new(),
        dels: { let mut s = BTreeSet::new(); s.insert("holding-plank".to_string()); s },
    });
    ops
}

let a_t = ops_de_base(&cases);
let ster = ops_avec_jeter(a_t.clone());
let a_t1 = ops_avec_radeau(&cases);
let t1s = ops_avec_jeter(a_t1.clone());

let vocabs: Vec<(&str, Vec<Op>)> = vec![
    ("A_t", a_t),
    ("A_t+discard", ster),
    ("A_t1 (+raft)", a_t1),
    ("A_t1+discard", t1s),
];

let buts: Vec<(&str, BTreeSet<String>)> = vec![
    ("g1: balise (5,2)", { let mut s = BTreeSet::new(); s.insert(prop(5, 2)); s }),
    ("g2: detenir le tresor", { let mut s = BTreeSet::new(); s.insert("holding-treasure".to_string()); s }),
    ("g3: detenir la cle", { let mut s = BTreeSet::new(); s.insert("holding-key".to_string()); s }),
];

fn py_list(items: &[&str]) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}

{
    let pi_lex = |o: &Op| (0u32, o.nom.clone());

    // Recalcule atteint (4 BFS complets)
    let mut atteint: HashMap<&str, HashMap<BTreeSet<String>, usize>> = HashMap::new();
    for (nom, v) in &vocabs {
        let (dist, _) = bfs_budget(&initial, v, None, &pi_lex);
        atteint.insert(nom, dist);
    }

    // base_buts depuis atteint["A_t"]
    let base_buts: BTreeSet<&str> = buts.iter()
        .filter(|(b, cond)| atteint["A_t"].keys().any(|s| cond.is_subset(s)))
        .map(|(b, _)| *b)
        .collect();

    for (nom, _) in &vocabs {
        if *nom == "A_t" {
            continue;
        }
        let ok: BTreeSet<&str> = buts.iter()
            .filter(|(b, cond)| atteint[nom].keys().any(|s| cond.is_subset(s)))
            .map(|(b, _)| *b)
            .collect();
        let nouveaux: BTreeSet<&str> = ok.difference(&base_buts).copied().collect();
        let perdus: BTreeSet<&str> = base_buts.difference(&ok).copied().collect();
        let nouveaux_str = if nouveaux.is_empty() {
            "AUCUN".to_string()
        } else {
            let mut prefixes: Vec<&str> = nouveaux.iter().map(|b| *b).map(|b| &b[..2]).collect();
            prefixes.sort();
            py_list(&prefixes)
        };
        let perdus_str = if perdus.is_empty() {
            "aucun".to_string()
        } else {
            let mut prefixes: Vec<&str> = perdus.iter().map(|b| *b).map(|b| &b[..2]).collect();
            prefixes.sort();
            py_list(&prefixes)
        };
        println!("A_t -> {:<16} nouveaux = {}   perdus = {}", nom, nouveaux_str, perdus_str);
    }
};

A_t -> A_t+discard      nouveaux = AUCUN   perdus = aucun


A_t -> A_t1 (+raft)     nouveaux = ['g1', 'g2']   perdus = aucun


A_t -> A_t1+discard     nouveaux = ['g1', 'g2']   perdus = aucun


### Lecture : $|\Delta A|$ décore, $\Delta(\text{buts})$ discrimine

Le résultat central du contrôle négatif : `A_t -> A_t+discard` fait croître $|A|$ de **36 à 54 états (+50 %)** sans rendre **aucun** but nouveau. Une mesure naïve de « progrès » par croissance d'espace ($|\Delta A| > 0$) validerait cet ajout — à tort. Le différentiel qui mesure est $\Delta(\text{buts atteignables})$, et lui reste **exactement nul**.

Verbatim de la sortie : « A_t -> A_t+discard nouveaux = AUCUN perdus = aucun » puis « A_t -> A_t1 (+raft) nouveaux = ['g1', 'g2'] perdus = aucun » — `discard` n'ajoute aucun but, `raft` en ajoute deux sans en perdre. La nuance est dans l'écart entre les deux colonnes : `discard` ne change pas les **buts**, mais il change bien $|A|$ (36 → 54 états) — l'opérateur ajoute des états, pas des objectifs atteignables.


La leçon pour la strate 7 : « élargir son espace » n'a de valeur que relativement à ce que l'agent **veut atteindre**. Un espace qui croît de 50 % sans ouvrir aucun but est du bruit, pas du progrès.

## $F$ et $r$ : le solveur a un budget

Le système n'est pas le solveur. Un BFS réel s'arrête à $B$ expansions — c'est le mur de Planners-10b (`BUDGET_NODES = 15 000`). Ici $F_B$ = découvert à budget $B$, $r = |F_B|/|A|$.

In [7]:
use std::collections::{BTreeSet, HashMap, VecDeque};

#[derive(Clone)]
struct Op {
    nom: String,
    precond: BTreeSet<String>,
    ajouts: BTreeSet<String>,
    dels: BTreeSet<String>,
}

impl Op {
    fn applicable(&self, etat: &BTreeSet<String>) -> bool {
        self.precond.is_subset(etat)
    }
}

fn successeurs(etat: &BTreeSet<String>, ops: &[Op], cle: &impl Fn(&Op) -> (u32, String)) -> Vec<(Op, BTreeSet<String>)> {
    let mut tried: Vec<&Op> = ops.iter().collect();
    tried.sort_by_key(|o| cle(o));
    let mut out = Vec::new();
    for op in tried {
        if op.applicable(etat) {
            let mut enfant = etat.clone();
            enfant.extend(op.ajouts.iter().cloned());
            for d in &op.dels {
                enfant.remove(d);
            }
            out.push((op.clone(), enfant));
        }
    }
    out
}

fn bfs_budget(initial: &BTreeSet<String>, ops: &[Op], budget: Option<usize>, cle: &impl Fn(&Op) -> (u32, String)) -> (HashMap<BTreeSet<String>, usize>, usize) {
    let mut dist: HashMap<BTreeSet<String>, usize> = HashMap::new();
    dist.insert(initial.clone(), 0);
    let mut file: VecDeque<BTreeSet<String>> = VecDeque::new();
    file.push_back(initial.clone());
    let mut expansions = 0;
    while !file.is_empty() && budget.map_or(true, |b| expansions < b) {
        let s = file.pop_front().unwrap();
        expansions += 1;
        for (_op, t) in successeurs(&s, ops, cle) {
            if !dist.contains_key(&t) {
                dist.insert(t.clone(), dist[&s] + 1);
                file.push_back(t);
            }
        }
    }
    (dist, expansions)
}

// Domaine
let cases: Vec<(i64, i64)> = {
    let cols_gauche = [0i64, 1, 2];
    let cols_droite = [4i64, 5];
    let rangs = [0i64, 1, 2];
    let mut s: BTreeSet<(i64, i64)> = BTreeSet::new();
    for &c in cols_gauche.iter().chain(cols_droite.iter()) {
        for &r in &rangs {
            s.insert((c, r));
        }
    }
    s.into_iter().collect()
};

fn prop(c: i64, r: i64) -> String {
    format!("at-x{}-y{}", c, r)
}

let initial: BTreeSet<String> = {
    let mut s = BTreeSet::new();
    s.insert(prop(0, 0));
    s.insert("plank-at-x1-y1".to_string());
    s.insert("key-at-x2-y0".to_string());
    s.insert("treasure-at-x4-y1".to_string());
    s
};

fn ops_de_base(cases: &[(i64, i64)]) -> Vec<Op> {
    let mut ops = Vec::new();
    for &(c, r) in cases {
        for (dc, dr, nom_dir) in [(1i64, 0i64, "est"), (-1, 0, "ouest"), (0, 1, "nord"), (0, -1, "sud")] {
            let cible = (c + dc, r + dr);
            if cases.contains(&cible) {
                ops.push(Op {
                    nom: format!("move-{}-x{}-y{}", nom_dir, c, r),
                    precond: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s },
                    ajouts: { let mut s = BTreeSet::new(); s.insert(prop(cible.0, cible.1)); s },
                    dels: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s },
                });
            }
        }
    }
    for (objet, cell) in [("plank", (1i64, 1i64)), ("key", (2i64, 0i64)), ("treasure", (4i64, 1i64))] {
        let (c, r) = cell;
        ops.push(Op {
            nom: format!("pickup-{}", objet),
            precond: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s.insert(format!("{}-at-x{}-y{}", objet, c, r)); s },
            ajouts: { let mut s = BTreeSet::new(); s.insert(format!("holding-{}", objet)); s },
            dels: { let mut s = BTreeSet::new(); s.insert(format!("{}-at-x{}-y{}", objet, c, r)); s },
        });
    }
    ops
}

fn ops_avec_radeau(cases: &[(i64, i64)]) -> Vec<Op> {
    let mut ops = ops_de_base(cases);
    for r in [0i64, 1, 2] {
        ops.push(Op {
            nom: format!("raft-across-y{}", r),
            precond: { let mut s = BTreeSet::new(); s.insert(prop(2, r)); s.insert("holding-plank".to_string()); s },
            ajouts: { let mut s = BTreeSet::new(); s.insert(prop(4, r)); s },
            dels: { let mut s = BTreeSet::new(); s.insert(prop(2, r)); s.insert("holding-plank".to_string()); s },
        });
    }
    ops
}

fn ops_avec_jeter(base: Vec<Op>) -> Vec<Op> {
    let mut ops = base;
    ops.push(Op {
        nom: "discard-plank".to_string(),
        precond: { let mut s = BTreeSet::new(); s.insert("holding-plank".to_string()); s },
        ajouts: BTreeSet::new(),
        dels: { let mut s = BTreeSet::new(); s.insert("holding-plank".to_string()); s },
    });
    ops
}

let a_t = ops_de_base(&cases);
let ster = ops_avec_jeter(a_t.clone());
let a_t1 = ops_avec_radeau(&cases);
let t1s = ops_avec_jeter(a_t1.clone());

let vocabs: Vec<(&str, Vec<Op>)> = vec![
    ("A_t", a_t),
    ("A_t+discard", ster),
    ("A_t1 (+raft)", a_t1),
    ("A_t1+discard", t1s),
];

let buts: Vec<(&str, BTreeSet<String>)> = vec![
    ("g1: balise (5,2)", { let mut s = BTreeSet::new(); s.insert(prop(5, 2)); s }),
    ("g2: detenir le tresor", { let mut s = BTreeSet::new(); s.insert("holding-treasure".to_string()); s }),
    ("g3: detenir la cle", { let mut s = BTreeSet::new(); s.insert("holding-key".to_string()); s }),
];

fn py_list(items: &[&str]) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}

{
    let pi_lex = |o: &Op| (0u32, o.nom.clone());

    // Denominateurs : BFS complets
    let mut atteint: HashMap<&str, HashMap<BTreeSet<String>, usize>> = HashMap::new();
    for (nom, v) in &vocabs {
        let (dist, _) = bfs_budget(&initial, v, None, &pi_lex);
        atteint.insert(nom, dist);
    }

    for B in [30usize, 60, 120] {
        println!("budget B = {} :", B);
        for (nom, v) in &vocabs {
            let (dist, _) = bfs_budget(&initial, v, Some(B), &pi_lex);
            let r = dist.len() as f64 / atteint[nom].len() as f64;
            let pct = format!("{:>5}", format!("{:.1}%", r * 100.0));
            let prefs: Vec<&str> = buts.iter()
                .filter(|(_, cond)| dist.keys().any(|s| cond.is_subset(s)))
                .map(|(b, _)| &b[..2])
                .collect();
            println!("  {:<16} |F_B| = {:>4}   r = {}   buts dans F : {}", nom, dist.len(), pct, py_list(&prefs));
        }
    }
};

budget B = 30 :


  A_t              |F_B| =   34   r = 94.4%   buts dans F : ['g3']


  A_t+discard      |F_B| =   38   r = 70.4%   buts dans F : ['g3']


  A_t1 (+raft)     |F_B| =   40   r = 66.7%   buts dans F : ['g1', 'g2', 'g3']


  A_t1+discard     |F_B| =   42   r = 53.8%   buts dans F : ['g2', 'g3']


budget B = 60 :


... (5 outputs left out)

  A_t              |F_B| =   36   r = 100.0%   buts dans F : ['g3']


  A_t+discard      |F_B| =   54   r = 100.0%   buts dans F : ['g3']


  A_t1 (+raft)     |F_B| =   60   r = 100.0%   buts dans F : ['g1', 'g2', 'g3']


  A_t1+discard     |F_B| =   78   r = 100.0%   buts dans F : ['g1', 'g2', 'g3']


### Lecture : l'élargissement non contrôlé **contracte** la liberté résiduelle

À budget $B=30$ : $r$ chute de **94,4 %** ($A_t$) à **53,8 %** ($A_{t1}$+discard). Le prix se paie deux fois : $r$ descend de **94,4 %** ($A_t$) à **66,7 %** ($A_{t1}$), puis à **53,8 %** ($A_{t1}$+discard). Le tableau réserve un piège : $A_{t1}$+discard affiche « buts dans F : ['g2', 'g3'] » — à budget serré, `discard` fait **perdre $g_1$** du front atteignable, alors même qu'il n'ajoutait aucun but. À $B=120$, $r$ remonte à **100,0 %** partout : la contraction n'est pas une propriété du système, c'est une propriété du **budget**.

Le quadruplet isole le paradoxe : élargir $A$ (le système peut plus) peut **réduire** $r$ (le solveur explore une part plus petite de ce qui existe). L'espace croît plus vite que le budget — c'est la face sombre de l'élargissement, invisible si on ne regarde que $A$.

## Le noyau : ce que la primitive ajoutée rend **inatteignable** (critère 4)

L'obligation méthodologique de #13565/#13568 : ne jamais conclure « élargissement » sans chercher le **noyau** — ce que l'ajout rend inatteignable. Deux niveaux à distinguer honnêtement :

1. **Noyau système** : dans STRIPS, les actions sont *optionnelles* — un ancien plan reste valide après l'ajout (monotonie de l'atteignabilité). Le noyau système est donc **vide** par construction : la mesure ci-dessus le confirme (`perdus = aucun` partout).
2. **Noyau à budget fixe** : le solveur, lui, n'a pas cette clémence. À $B$ constant, élargir le vocabulaire dilue le budget sur plus de successeurs — des buts qui tenaient dans $F$ peuvent en sortir.

In [8]:
use std::collections::{BTreeSet, HashMap, VecDeque};

#[derive(Clone)]
struct Op {
    nom: String,
    precond: BTreeSet<String>,
    ajouts: BTreeSet<String>,
    dels: BTreeSet<String>,
}

impl Op {
    fn applicable(&self, etat: &BTreeSet<String>) -> bool {
        self.precond.is_subset(etat)
    }
}

fn successeurs(etat: &BTreeSet<String>, ops: &[Op], cle: &impl Fn(&Op) -> (u32, String)) -> Vec<(Op, BTreeSet<String>)> {
    let mut tried: Vec<&Op> = ops.iter().collect();
    tried.sort_by_key(|o| cle(o));
    let mut out = Vec::new();
    for op in tried {
        if op.applicable(etat) {
            let mut enfant = etat.clone();
            enfant.extend(op.ajouts.iter().cloned());
            for d in &op.dels {
                enfant.remove(d);
            }
            out.push((op.clone(), enfant));
        }
    }
    out
}

fn bfs_budget(initial: &BTreeSet<String>, ops: &[Op], budget: Option<usize>, cle: &impl Fn(&Op) -> (u32, String)) -> (HashMap<BTreeSet<String>, usize>, usize) {
    let mut dist: HashMap<BTreeSet<String>, usize> = HashMap::new();
    dist.insert(initial.clone(), 0);
    let mut file: VecDeque<BTreeSet<String>> = VecDeque::new();
    file.push_back(initial.clone());
    let mut expansions = 0;
    while !file.is_empty() && budget.map_or(true, |b| expansions < b) {
        let s = file.pop_front().unwrap();
        expansions += 1;
        for (_op, t) in successeurs(&s, ops, cle) {
            if !dist.contains_key(&t) {
                dist.insert(t.clone(), dist[&s] + 1);
                file.push_back(t);
            }
        }
    }
    (dist, expansions)
}

// Domaine
let cases: Vec<(i64, i64)> = {
    let cols_gauche = [0i64, 1, 2];
    let cols_droite = [4i64, 5];
    let rangs = [0i64, 1, 2];
    let mut s: BTreeSet<(i64, i64)> = BTreeSet::new();
    for &c in cols_gauche.iter().chain(cols_droite.iter()) {
        for &r in &rangs {
            s.insert((c, r));
        }
    }
    s.into_iter().collect()
};

fn prop(c: i64, r: i64) -> String {
    format!("at-x{}-y{}", c, r)
}

let initial: BTreeSet<String> = {
    let mut s = BTreeSet::new();
    s.insert(prop(0, 0));
    s.insert("plank-at-x1-y1".to_string());
    s.insert("key-at-x2-y0".to_string());
    s.insert("treasure-at-x4-y1".to_string());
    s
};

fn ops_de_base(cases: &[(i64, i64)]) -> Vec<Op> {
    let mut ops = Vec::new();
    for &(c, r) in cases {
        for (dc, dr, nom_dir) in [(1i64, 0i64, "est"), (-1, 0, "ouest"), (0, 1, "nord"), (0, -1, "sud")] {
            let cible = (c + dc, r + dr);
            if cases.contains(&cible) {
                ops.push(Op {
                    nom: format!("move-{}-x{}-y{}", nom_dir, c, r),
                    precond: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s },
                    ajouts: { let mut s = BTreeSet::new(); s.insert(prop(cible.0, cible.1)); s },
                    dels: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s },
                });
            }
        }
    }
    for (objet, cell) in [("plank", (1i64, 1i64)), ("key", (2i64, 0i64)), ("treasure", (4i64, 1i64))] {
        let (c, r) = cell;
        ops.push(Op {
            nom: format!("pickup-{}", objet),
            precond: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s.insert(format!("{}-at-x{}-y{}", objet, c, r)); s },
            ajouts: { let mut s = BTreeSet::new(); s.insert(format!("holding-{}", objet)); s },
            dels: { let mut s = BTreeSet::new(); s.insert(format!("{}-at-x{}-y{}", objet, c, r)); s },
        });
    }
    ops
}

fn ops_avec_radeau(cases: &[(i64, i64)]) -> Vec<Op> {
    let mut ops = ops_de_base(cases);
    for r in [0i64, 1, 2] {
        ops.push(Op {
            nom: format!("raft-across-y{}", r),
            precond: { let mut s = BTreeSet::new(); s.insert(prop(2, r)); s.insert("holding-plank".to_string()); s },
            ajouts: { let mut s = BTreeSet::new(); s.insert(prop(4, r)); s },
            dels: { let mut s = BTreeSet::new(); s.insert(prop(2, r)); s.insert("holding-plank".to_string()); s },
        });
    }
    ops
}

fn ops_avec_jeter(base: Vec<Op>) -> Vec<Op> {
    let mut ops = base;
    ops.push(Op {
        nom: "discard-plank".to_string(),
        precond: { let mut s = BTreeSet::new(); s.insert("holding-plank".to_string()); s },
        ajouts: BTreeSet::new(),
        dels: { let mut s = BTreeSet::new(); s.insert("holding-plank".to_string()); s },
    });
    ops
}

let a_t = ops_de_base(&cases);
let ster = ops_avec_jeter(a_t.clone());
let a_t1 = ops_avec_radeau(&cases);
let t1s = ops_avec_jeter(a_t1.clone());

let vocabs: Vec<(&str, Vec<Op>)> = vec![
    ("A_t", a_t),
    ("A_t+discard", ster),
    ("A_t1 (+raft)", a_t1),
    ("A_t1+discard", t1s),
];

let buts: Vec<(&str, BTreeSet<String>)> = vec![
    ("g1: balise (5,2)", { let mut s = BTreeSet::new(); s.insert(prop(5, 2)); s }),
    ("g2: detenir le tresor", { let mut s = BTreeSet::new(); s.insert("holding-treasure".to_string()); s }),
    ("g3: detenir la cle", { let mut s = BTreeSet::new(); s.insert("holding-key".to_string()); s }),
];

// Expansions cumulees pour la premiere decouverte du but (BFS ordre pi)
fn expansions_pour_but(
    initial: &BTreeSet<String>,
    ops: &[Op],
    but: &BTreeSet<String>,
    cle: &impl Fn(&Op) -> (u32, String),
) -> (Option<usize>, Option<usize>) {
    let mut dist: HashMap<BTreeSet<String>, usize> = HashMap::new();
    dist.insert(initial.clone(), 0);
    let mut file: VecDeque<BTreeSet<String>> = VecDeque::new();
    file.push_back(initial.clone());
    let mut expansions = 0;
    while let Some(s) = file.pop_front() {
        expansions += 1;
        if but.is_subset(&s) {
            return (Some(expansions), Some(dist[&s]));
        }
        for (_op, t) in successeurs(&s, ops, cle) {
            if !dist.contains_key(&t) {
                dist.insert(t.clone(), dist[&s] + 1);
                file.push_back(t);
            }
        }
    }
    (None, None)
}

fn py_opt(o: Option<i64>) -> String {
    match o {
        None => "None".to_string(),
        Some(x) => x.to_string(),
    }
}

{
    let pi_lex = |o: &Op| (0u32, o.nom.clone());

    println!("Expansions necessaires a la premiere decouverte de chaque but :");
    for (b_nom, b_cond) in &buts {
        let ligne = vocabs.iter()
            .map(|(nom, v)| format!("{}={}", nom, py_opt(expansions_pour_but(&initial, v, b_cond, &pi_lex).0.map(|x| x as i64))))
            .collect::<Vec<_>>()
            .join("  ");
        println!("  {:<24} {}", b_nom, ligne);
    }
};

Expansions necessaires a la premiere decouverte de chaque but :


  g1: balise (5,2)         A_t=None  A_t+discard=None  A_t1 (+raft)=39  A_t1+discard=50


  g2: detenir le tresor    A_t=None  A_t+discard=None  A_t1 (+raft)=33  A_t1+discard=42


  g3: detenir la cle       A_t=8  A_t+discard=8  A_t1 (+raft)=8  A_t1+discard=8


### Lecture : le noyau non vide vit à budget fixe

Verbatim des expansions à la première découverte : $g_1$ « A_t=None A_t+discard=None A_t1 (+raft)=39 A_t1+discard=50 », $g_2$ « None None 33 42 », $g_3$ « 8 8 8 8 » — $g_1$ et $g_2$ exigent le raft (indisponible → `None`), $g_3$ est atteint partout en 8 expansions. La nécessité du raft se lit sur deux lignes, l'indifférence sur la troisième.


- $g_1$ : **inatteignable** dans $A_t$ et $A_t$+discard (le gouffre), découvert à **39** expansions dans $A_{t1}$ — mais à **50** dans $A_{t1}$+discard ;
- $g_2$ : **33** dans $A_{t1}$ contre **42** dans $A_{t1}$+discard.

Lecture directe sur le tableau $F/r$ à $B=30$ : $g_1$ est **dans** $F(A_{t1})$ mais **hors** $F(A_{t1}$+discard$)$ — l'ajout de `discard` (une primitive qui ne sert à *rien*) a rendu $g_1$ inatteignable **au solveur à budget constant**. C'est le noyau : pas au système (monotonie), mais au solveur (dilution). Les deux niveaux sont mesurés et distingués — c'est exactement ce que le critère 4 exige.

## Cas 3 — navigation : seul $\pi$ bouge (le bras Planners-10b)

Troisième signature : $A$ et $F$ **inchangés**, seule la **politique** change. C'est la position de [Planners-10b](../../SymbolicAI/Planners/04-NeuroSymbolic/Planners-10b-LLM-Space-Reducer-Python.ipynb) : le LLM ne résout pas, il **réduit** l'espace exploré en premier (quelles primitives essayer d'abord) ; le solving reste déterministe. Ici, la version jouet : réordonner l'expansion — `pickup` d'abord, ou `raft` d'abord, contre l'ordre lexicographique.

In [9]:
use std::collections::{BTreeSet, HashMap, VecDeque};

#[derive(Clone)]
struct Op {
    nom: String,
    precond: BTreeSet<String>,
    ajouts: BTreeSet<String>,
    dels: BTreeSet<String>,
}

impl Op {
    fn applicable(&self, etat: &BTreeSet<String>) -> bool {
        self.precond.is_subset(etat)
    }
}

fn successeurs(etat: &BTreeSet<String>, ops: &[Op], cle: &impl Fn(&Op) -> (u32, String)) -> Vec<(Op, BTreeSet<String>)> {
    let mut tried: Vec<&Op> = ops.iter().collect();
    tried.sort_by_key(|o| cle(o));
    let mut out = Vec::new();
    for op in tried {
        if op.applicable(etat) {
            let mut enfant = etat.clone();
            enfant.extend(op.ajouts.iter().cloned());
            for d in &op.dels {
                enfant.remove(d);
            }
            out.push((op.clone(), enfant));
        }
    }
    out
}

fn bfs_budget(initial: &BTreeSet<String>, ops: &[Op], budget: Option<usize>, cle: &impl Fn(&Op) -> (u32, String)) -> (HashMap<BTreeSet<String>, usize>, usize) {
    let mut dist: HashMap<BTreeSet<String>, usize> = HashMap::new();
    dist.insert(initial.clone(), 0);
    let mut file: VecDeque<BTreeSet<String>> = VecDeque::new();
    file.push_back(initial.clone());
    let mut expansions = 0;
    while !file.is_empty() && budget.map_or(true, |b| expansions < b) {
        let s = file.pop_front().unwrap();
        expansions += 1;
        for (_op, t) in successeurs(&s, ops, cle) {
            if !dist.contains_key(&t) {
                dist.insert(t.clone(), dist[&s] + 1);
                file.push_back(t);
            }
        }
    }
    (dist, expansions)
}

// Domaine
let cases: Vec<(i64, i64)> = {
    let cols_gauche = [0i64, 1, 2];
    let cols_droite = [4i64, 5];
    let rangs = [0i64, 1, 2];
    let mut s: BTreeSet<(i64, i64)> = BTreeSet::new();
    for &c in cols_gauche.iter().chain(cols_droite.iter()) {
        for &r in &rangs {
            s.insert((c, r));
        }
    }
    s.into_iter().collect()
};

fn prop(c: i64, r: i64) -> String {
    format!("at-x{}-y{}", c, r)
}

let initial: BTreeSet<String> = {
    let mut s = BTreeSet::new();
    s.insert(prop(0, 0));
    s.insert("plank-at-x1-y1".to_string());
    s.insert("key-at-x2-y0".to_string());
    s.insert("treasure-at-x4-y1".to_string());
    s
};

fn ops_de_base(cases: &[(i64, i64)]) -> Vec<Op> {
    let mut ops = Vec::new();
    for &(c, r) in cases {
        for (dc, dr, nom_dir) in [(1i64, 0i64, "est"), (-1, 0, "ouest"), (0, 1, "nord"), (0, -1, "sud")] {
            let cible = (c + dc, r + dr);
            if cases.contains(&cible) {
                ops.push(Op {
                    nom: format!("move-{}-x{}-y{}", nom_dir, c, r),
                    precond: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s },
                    ajouts: { let mut s = BTreeSet::new(); s.insert(prop(cible.0, cible.1)); s },
                    dels: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s },
                });
            }
        }
    }
    for (objet, cell) in [("plank", (1i64, 1i64)), ("key", (2i64, 0i64)), ("treasure", (4i64, 1i64))] {
        let (c, r) = cell;
        ops.push(Op {
            nom: format!("pickup-{}", objet),
            precond: { let mut s = BTreeSet::new(); s.insert(prop(c, r)); s.insert(format!("{}-at-x{}-y{}", objet, c, r)); s },
            ajouts: { let mut s = BTreeSet::new(); s.insert(format!("holding-{}", objet)); s },
            dels: { let mut s = BTreeSet::new(); s.insert(format!("{}-at-x{}-y{}", objet, c, r)); s },
        });
    }
    ops
}

fn ops_avec_radeau(cases: &[(i64, i64)]) -> Vec<Op> {
    let mut ops = ops_de_base(cases);
    for r in [0i64, 1, 2] {
        ops.push(Op {
            nom: format!("raft-across-y{}", r),
            precond: { let mut s = BTreeSet::new(); s.insert(prop(2, r)); s.insert("holding-plank".to_string()); s },
            ajouts: { let mut s = BTreeSet::new(); s.insert(prop(4, r)); s },
            dels: { let mut s = BTreeSet::new(); s.insert(prop(2, r)); s.insert("holding-plank".to_string()); s },
        });
    }
    ops
}

fn ops_avec_jeter(base: Vec<Op>) -> Vec<Op> {
    let mut ops = base;
    ops.push(Op {
        nom: "discard-plank".to_string(),
        precond: { let mut s = BTreeSet::new(); s.insert("holding-plank".to_string()); s },
        ajouts: BTreeSet::new(),
        dels: { let mut s = BTreeSet::new(); s.insert("holding-plank".to_string()); s },
    });
    ops
}

let a_t = ops_de_base(&cases);
let ster = ops_avec_jeter(a_t.clone());
let a_t1 = ops_avec_radeau(&cases);
let t1s = ops_avec_jeter(a_t1.clone());

let vocabs: Vec<(&str, Vec<Op>)> = vec![
    ("A_t", a_t),
    ("A_t+discard", ster),
    ("A_t1 (+raft)", a_t1.clone()),
    ("A_t1+discard", t1s),
];

let buts: Vec<(&str, BTreeSet<String>)> = vec![
    ("g1: balise (5,2)", { let mut s = BTreeSet::new(); s.insert(prop(5, 2)); s }),
    ("g2: detenir le tresor", { let mut s = BTreeSet::new(); s.insert("holding-treasure".to_string()); s }),
    ("g3: detenir la cle", { let mut s = BTreeSet::new(); s.insert("holding-key".to_string()); s }),
];

fn expansions_pour_but(
    initial: &BTreeSet<String>,
    ops: &[Op],
    but: &BTreeSet<String>,
    cle: &impl Fn(&Op) -> (u32, String),
) -> (Option<usize>, Option<usize>) {
    let mut dist: HashMap<BTreeSet<String>, usize> = HashMap::new();
    dist.insert(initial.clone(), 0);
    let mut file: VecDeque<BTreeSet<String>> = VecDeque::new();
    file.push_back(initial.clone());
    let mut expansions = 0;
    while let Some(s) = file.pop_front() {
        expansions += 1;
        if but.is_subset(&s) {
            return (Some(expansions), Some(dist[&s]));
        }
        for (_op, t) in successeurs(&s, ops, cle) {
            if !dist.contains_key(&t) {
                dist.insert(t.clone(), dist[&s] + 1);
                file.push_back(t);
            }
        }
    }
    (None, None)
}

{
    // Trois politiques de tri
    let pi_lex = |o: &Op| (0u32, o.nom.clone());
    let pi_pickup = |o: &Op| (if o.nom.starts_with("pickup") { 0 } else { 1 }, o.nom.clone());
    let pi_raft = |o: &Op| (if o.nom.starts_with("raft") { 0 } else { 1 }, o.nom.clone());

    println!("A_t1, trois politiques -- A et F inchanges par construction (meme dynamique) :");
    println!("{:<24} {:>8} {:>10} {:>8}   h*", "but", "pi_lex", "pi_pickup", "pi_raft");
    for (b_nom, b_cond) in &buts {
        let res = [
            expansions_pour_but(&initial, &a_t1, b_cond, &pi_lex),
            expansions_pour_but(&initial, &a_t1, b_cond, &pi_pickup),
            expansions_pour_but(&initial, &a_t1, b_cond, &pi_raft),
        ];
        println!("  {:<22} {:>7} {:>9} {:>7}   {} (inchange)", b_nom, res[0].0.unwrap(), res[1].0.unwrap(), res[2].0.unwrap(), res[0].1.unwrap());
    }
};

A_t1, trois politiques -- A et F inchanges par construction (meme dynamique) :


but                        pi_lex  pi_pickup  pi_raft   h*


  g1: balise (5,2)            39        39      39   7 (inchange)


  g2: detenir le tresor       33        32      32   6 (inchange)


  g3: detenir la cle           8         7       8   3 (inchange)


### Lecture : trois signatures, un seul quadruplet

$\pi_{pickup}$ améliore la découverte de $g_2$ (33 → 32) et $g_3$ (8 → 7) ; $h^*$ est **inchangé** partout (la distance optimale ne dépend pas de l'ordre d'exploration — garanti par le lake : l'admissibilité de $h^+$ ne connaît pas $\pi$). La table des trois cas de la strate 7, sur le même domaine :

| Cas | $A$ | $F_B$ | $r$ | Témoin mesuré |
|---|---|---|---|---|
| **Élargissement réel** | croît (36→60) | croît | varie | `+raft` : 1/3 → 3/3 buts |
| **Dilution (face sombre)** | croît (60→78) | stagne à budget | **chute** (66,7 %→53,8 %) | `+discard` : $g_1$ sort de $F$ à $B{=}30$ |
| **Navigation** | **inchangé** | **inchangé** | inchangé | $\pi_{pickup}$ : exp($g_3$) 8 → 7, $h^*$ stable |

« L'agent a progressé » ne dit rien tant que la ligne du tableau n'est pas nommée. C'est la réponse du quadruplet à la formule nébuleuse.

## À retenir

*Planners-5c mesurait le différentiel d'atteignabilité mais n'avait pas le contrôle négatif **exécuté** (exercice 2 non résolu), ni le noyau, ni $r$. Planners-10b montrait la réduction LLM sur un autre domaine (grilles ARC-like), sans lien avec l'élargissement. La greffe :*

1. **nomme** le quadruplet $(A, F, r, \pi)$ — les quatre objets, mesurables séparément ;
2. **exécute** le contrôle négatif : +50 % d'états, 0 but nouveau — la croissance d'espace n'est pas le progrès ;
3. **mesure le noyau** à deux niveaux : vide au système (monotonie STRIPS, confirmée), non vide au solveur à budget ;
4. **discrimine les trois cas** de la strate 7 sur le même banc — le témoin de contraction *certifiée* (#13565, Beck-Fiala) viendra greffer le quatrième quand livré.

## Limites

- Le domaine est **petit** (36-78 états) : les effets de budget sont visibles à $B \in [30, 120]$, pas à l'échelle du mur de Planners-10b (15 000 nœuds). La structure des trois signatures est identique ; les magnitudes ne se transfèrent pas linéairement.
- La monotonie STRIPS (noyau système vide) est une propriété du **cadre** (actions optionnelles, sans contraintes globales) : un domaine à invariants (capacité, Beck-Fiala) peut avoir un noyau système non vide — c'est l'exercice 3.
- $\pi$ jouet : trois ordres statiques. Le réducteur de Planners-10b est *par tâche* (le LLM propose le sous-ensemble pertinent) — une politique dépendante de l'état, plus riche que nos clés de tri constantes.
- Les proxys $|\Delta A|$, $r$ sont des **bornes d'exploration**, pas des prédicteurs de performance d'agent complet (pas de récompense, pas d'apprentissage ici).

## Vérifiez votre compréhension

1. Le contrôle négatif exécuté donne « +50 % d'états, 0 but nouveau » : que prouve-t-il ?
2. Le noyau est mesuré à deux niveaux — vide au système, non vide au solveur à budget : pourquoi les distinguer ?
3. La monotonie STRIPS est dite « une propriété du cadre » : à quelle condition le noyau système devient-il non vide ?

<details>
<summary>Réponses</summary>

1. Que **la croissance d'espace n'est pas le progrès** : |ΔA| est une borne d'exploration, et un élargissement qui n'ouvre aucun but atteignable nouveau est un ajout inutile. L'exercice 1 pousse la logique un cran plus loin : avec `burn-plank`, l'état-piège rend l'ajout **nuisible** — inutile et nuisible sont deux échecs distincts de la greffe.
2. La monotonie STRIPS garantit qu'aucun ajout ne rend un but inatteignable — le noyau **système** est vide par propriété du cadre, et le carnet le confirme. Mais le solveur à budget ($B \in [30, 120]$) peut ne pas trouver dans son quota : cette contraction **apparente** est un artefact de budget, pas de sémantique. Sans les deux niveaux, on lirait une limite de calcul comme une perte réelle d'atteignabilité.
3. Quand le domaine porte des **invariants** : la monotonie tient aux actions optionnelles sans contraintes globales ; un domaine à contrainte de capacité (l'invariant carry-capacity-1 / main-libre de l'exercice 3, ou Beck-Fiala) peut rendre un but atteignable inatteignable après greffe — le noyau système devient non vide. C'est le quatrième cas que le témoin de contraction *certifiée* (#13565) viendra greffer sur ce banc.

</details>

In [10]:
// Exercice 1 a completer : la primitive qui piege reellement.
// Ajouter burn-plank (detruire la planchette AVANT pickup : etat-piege plus precoce
// que discard -- preconditions {plank-at-x1-y1} sans holding), construire A_t1+burn,
// mesurer : (a) |A| croit-il davantage qu'avec discard ? (b) exp(g1) et exp(g2) :
// le noyau solveur croit-il ? (c) r a B=30 : conclusion sur la difference
// entre "ajout inutile" et "ajout nuisible".
println!("Exercice a completer");
let _resultat: Option<i64> = None; // TODO etudiant

Exercice a completer


### Exercice 2 — La politique dégradante

Construire $\pi_{degrade}$ qui fait **pire** que l'ordre lexicographique sur au moins un but (indice : mettre les `move` inutiles en tête, ou trier par nom décroissant). Mesurer la dégradation, et vérifier que $h^*$ reste inchangé — la politique ne peut dégrader la *découverte*, jamais la *distance*.

In [11]:
// Exercice 2 a completer : pi_degrade = lambda o: ... ; mesurer
// expansions_pour_but(INITIAL, A_T1, BUTS[...], pi_degrade) sur les 3 buts,
// comparer a pi_lex, confirmer h* inchange.
println!("Exercice a completer");
let _resultat: Option<i64> = None; // TODO etudiant

Exercice a completer


### Exercice 3 — Le noyau **système** non vide : la contrainte d'invariant

Le noyau système est vide en STRIPS pur (actions optionnelles). Le rendre non vide : ajouter un **invariant global** — par exemple `carry-capacity-1` (on ne peut tenir qu'un objet à la fois : chaque `pickup-x` a en précondition `hand-free`, chaque `pickup-y` supprime `hand-free`). Construire ce vocabulaire $A_{t1}^{cap}$ et montrer qu'un but **atteignable dans $A_{t1}$** devient inatteignable : le noyau système, cette fois, est mesuré non vide. C'est le pont vers la contraction **certifiée** de #13565 (Beck-Fiala).

In [12]:
// Exercice 3 a completer : construire A_t1_cap (invariant hand-free sur les pickups),
// mesurer les buts atteignables, identifier le but perdu vs A_t1 -- le noyau systeme.
println!("Exercice a completer");
let _resultat: Option<i64> = None; // TODO etudiant

Exercice a completer


## Pour aller plus loin

- Issue #13568 — la greffe 2 : quadruplet $(A, F, r, \pi)$, contrôle négatif, noyau.
- [Planners-5c — Le différentiel d'atteignabilité](../../SymbolicAI/Planners/02-Classical/Planners-05c-Differentiel-Atteignabilite-Python.ipynb) — le socle STRIPS et le domaine, copiés fidèlement.
- [Planners-10b — Le LLM comme réducteur d'espace](../../SymbolicAI/Planners/04-NeuroSymbolic/Planners-10b-LLM-Space-Reducer-Python.ipynb) — le bras restriction, moteur [aicpp](https://github.com/Julien-Livet/aicpp) (Julien Livet, Apache 2.0).
- #13565 — discrepancy_lean / Beck-Fiala : le témoin de contraction certifiée (à greffer).
- Lake `planning_lean` : `Relaxation.lean:56`, `Strips.lean:58`, `Admissibility.lean:41` — les trois garanties citées.

- **[ICT-Greffe4 — Le vote argumenté sur chaîne](ICT-Greffe4-VoteOnChain-Python.ipynb)** : fermer la boucle argumentation, choix social et exécution on-chain.
- **[ICT-0 — Cadrage de la série](ICT-0-Framing.md)** : le principe directeur de la série ICT.

> Carnet issu de l'EPIC #26534.